# Build Event Windows

This notebook aligns stock returns and market returns around each earnings call event.

The event anchor is:

- `event_trading_day_final`

All windows are defined in **trading time**, not calendar time.

This notebook constructs the event-time panel needed for:

- market model estimation window: `[-120, -20]`
- CAR windows: `[0,1]` and `[0,3]`
- volatility windows: `[-10,-1]` and `[+1,+10]`

The notebook does **not** compute abnormal returns or CAR yet. It only creates the aligned event-window dataset used in later steps.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Define file paths

In [2]:
metadata_path = Path("../data/processed/event_metadata_final.csv")
prices_path = Path("../data/processed/market_data_with_returns.csv")
index_path = Path("../data/processed/index_returns.csv")

output_path = Path("../data/processed/event_windows.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

## 2. Load datasets

We load:
- event metadata
- stock returns
- NASDAQ index returns

In [3]:
metadata = pd.read_csv(metadata_path)
prices = pd.read_csv(prices_path)
index_df = pd.read_csv(index_path)

print("metadata shape:", metadata.shape)
print("prices shape:", prices.shape)
print("index shape:", index_df.shape)

metadata shape: (188, 15)
prices shape: (13160, 9)
index shape: (1316, 8)


## 3. Parse dates and sort data

Sorting is essential because event windows are based on trading-day order.

In [4]:
metadata["event_trading_day_final"] = pd.to_datetime(
    metadata["event_trading_day_final"], errors="coerce"
)
prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce")
index_df["Date"] = pd.to_datetime(index_df["Date"], errors="coerce")

metadata = metadata.sort_values(["ticker", "event_trading_day_final"]).reset_index(drop=True)
prices = prices.sort_values(["ticker", "Date"]).reset_index(drop=True)
index_df = index_df.sort_values("Date").reset_index(drop=True)

## 4. Keep only columns needed for event-window construction

This keeps the pipeline clean and makes downstream merges easier to inspect.

In [5]:
metadata_cols = [
    "ticker",
    "file_name",
    "event_trading_day_final",
    "call_datetime_et",
    "after_market_close_et",
]

available_metadata_cols = [c for c in metadata_cols if c in metadata.columns]
metadata = metadata[available_metadata_cols].copy()

price_cols = ["Date", "ticker", "Adj Close", "return"]
prices = prices[price_cols].copy()

index_cols = ["Date", "Adj Close", "return"]
index_df = index_df[index_cols].copy()
index_df = index_df.rename(
    columns={
        "Adj Close": "index_adj_close",
        "return": "market_return",
    }
)

display(metadata.head())
display(prices.head())
display(index_df.head())

,ticker,file_name,event_trading_day_final,call_datetime_et,after_market_close_et
0,AAPL,2016-Jan-26-AAPL.txt,2016-01-27,2016-01-26T17:00:00-05:00,True
1,AAPL,2016-Apr-26-AAPL.txt,2016-04-27,2016-04-26T17:00:00-04:00,True
2,AAPL,2016-Jul-26-AAPL.txt,2016-07-27,2016-07-26T17:00:00-04:00,True
3,AAPL,2016-Oct-25-AAPL.txt,2016-10-26,2016-10-25T17:00:00-04:00,True
4,AAPL,2017-Jan-31-AAPL.txt,2017-02-01,2017-01-31T17:00:00-05:00,True


,Date,ticker,Adj Close,return
0,2015-06-30,AAPL,28.006931,0.007227
1,2015-07-01,AAPL,28.268185,0.009328
2,2015-07-02,AAPL,28.232452,-0.001264
3,2015-07-06,AAPL,28.134211,-0.003480
4,2015-07-07,AAPL,28.064987,-0.002460


,Date,index_adj_close,market_return
0,2015-06-30,4986.870117,0.005728
1,2015-07-01,5013.120117,0.005264
2,2015-07-02,5009.209961,-0.000780
3,2015-07-06,4991.939941,-0.003448
4,2015-07-07,4997.459961,0.001106


## 5. Add trading-day index within each ticker

For each stock, we create an integer trading-day position.

This lets us identify the event day by row position and then move backward or forward in trading time.

In [6]:
prices["trading_idx"] = prices.groupby("ticker").cumcount()
display(prices.head(10))

,Date,ticker,Adj Close,return,trading_idx
0,2015-06-30,AAPL,28.006931,0.007227,0
1,2015-07-01,AAPL,28.268185,0.009328,1
2,2015-07-02,AAPL,28.232452,-0.001264,2
3,2015-07-06,AAPL,28.134211,-0.003480,3
4,2015-07-07,AAPL,28.064987,-0.002460,4
5,2015-07-08,AAPL,27.368326,-0.024823,5
6,2015-07-09,AAPL,26.810120,-0.020396,6
7,2015-07-10,AAPL,27.526863,0.026734,7
8,2015-07-13,AAPL,28.058290,0.019306,8
9,2015-07-14,AAPL,28.047123,-0.000398,9


## 6. Merge market returns onto stock returns by date

Each stock-date observation should also include the NASDAQ market return for the same trading day.

In [7]:
prices = prices.merge(
    index_df[["Date", "index_adj_close", "market_return"]],
    on="Date",
    how="left",
    validate="many_to_one",
)

print("Missing market_return after merge:", prices["market_return"].isna().sum())
assert prices["market_return"].notna().all(), "Some stock rows are missing market returns after merge"

display(prices.head())

Missing market_return after merge: 0


,Date,ticker,Adj Close,return,trading_idx,index_adj_close,market_return
0,2015-06-30,AAPL,28.006931,0.007227,0,4986.870117,0.005728
1,2015-07-01,AAPL,28.268185,0.009328,1,5013.120117,0.005264
2,2015-07-02,AAPL,28.232452,-0.001264,2,5009.209961,-0.000780
3,2015-07-06,AAPL,28.134211,-0.003480,3,4991.939941,-0.003448
4,2015-07-07,AAPL,28.064987,-0.002460,4,4997.459961,0.001106


## 7. Create an event identifier

Each event gets a unique ID so the same ticker can appear multiple times across different earnings calls.

In [8]:
metadata = metadata.reset_index(drop=True)
metadata["event_id"] = metadata.index + 1
display(metadata.head())

,ticker,file_name,event_trading_day_final,call_datetime_et,after_market_close_et,event_id
0,AAPL,2016-Jan-26-AAPL.txt,2016-01-27,2016-01-26T17:00:00-05:00,True,1
1,AAPL,2016-Apr-26-AAPL.txt,2016-04-27,2016-04-26T17:00:00-04:00,True,2
2,AAPL,2016-Jul-26-AAPL.txt,2016-07-27,2016-07-26T17:00:00-04:00,True,3
3,AAPL,2016-Oct-25-AAPL.txt,2016-10-26,2016-10-25T17:00:00-04:00,True,4
4,AAPL,2017-Jan-31-AAPL.txt,2017-02-01,2017-01-31T17:00:00-05:00,True,5


## 8. Match each event to its stock trading-day row

We merge event metadata to stock returns using:
- `ticker`
- `event_trading_day_final` ↔ `Date`

This identifies the stock observation corresponding to event day `t = 0`.

In [9]:
event_matches = metadata.merge(
    prices[["ticker", "Date", "trading_idx"]],
    left_on=["ticker", "event_trading_day_final"],
    right_on=["ticker", "Date"],
    how="left",
    validate="one_to_one",
)

print("Events:", len(event_matches))
print("Events missing trading_idx:", event_matches["trading_idx"].isna().sum())

display(event_matches.head())

Events: 188
Events missing trading_idx: 0


,ticker,file_name,event_trading_day_final,call_datetime_et,after_market_close_et,event_id,Date,trading_idx
0,AAPL,2016-Jan-26-AAPL.txt,2016-01-27,2016-01-26T17:00:00-05:00,True,1,2016-01-27,145
1,AAPL,2016-Apr-26-AAPL.txt,2016-04-27,2016-04-26T17:00:00-04:00,True,2,2016-04-27,208
2,AAPL,2016-Jul-26-AAPL.txt,2016-07-27,2016-07-26T17:00:00-04:00,True,3,2016-07-27,271
3,AAPL,2016-Oct-25-AAPL.txt,2016-10-26,2016-10-25T17:00:00-04:00,True,4,2016-10-26,335
4,AAPL,2017-Jan-31-AAPL.txt,2017-02-01,2017-01-31T17:00:00-05:00,True,5,2017-02-01,401


## 9. Validate event-day matching

All events should match exactly one stock return row on the event trading day.

In [10]:
missing_event_matches = event_matches[event_matches["trading_idx"].isna()].copy()

print("Missing event matches:", len(missing_event_matches))
display(missing_event_matches.head(20))

assert len(missing_event_matches) == 0, "Some events could not be matched to stock returns on event_trading_day_final"

Missing event matches: 0


,ticker,file_name,event_trading_day_final,call_datetime_et,after_market_close_et,event_id,Date,trading_idx


## 10. Define the maximum event window needed

The report requires:
- earliest day: `-120`
- latest day: `+10`

We therefore extract the full window `[-120, +10]` for each event and label relative time as `t`.

In [11]:
WINDOW_START = -120
WINDOW_END = 10
print("Full extraction window:", WINDOW_START, "to", WINDOW_END)

Full extraction window: -120 to 10


## 11. Build event windows

For each event:
1. find the event trading-day index
2. extract stock rows from `t = -120` to `t = +10`
3. create a relative trading-time variable `t`
4. attach event metadata to each row

In [12]:
event_windows = []

for _, event in event_matches.iterrows():
    ticker = event["ticker"]
    event_id = event["event_id"]
    event_date = event["event_trading_day_final"]
    event_trading_idx = int(event["trading_idx"])

    ticker_prices = prices[prices["ticker"] == ticker].copy()

    window_start_idx = event_trading_idx + WINDOW_START
    window_end_idx = event_trading_idx + WINDOW_END

    window_df = ticker_prices[
        (ticker_prices["trading_idx"] >= window_start_idx) &
        (ticker_prices["trading_idx"] <= window_end_idx)
    ].copy()

    window_df["event_id"] = event_id
    window_df["event_trading_day_final"] = event_date
    window_df["t"] = window_df["trading_idx"] - event_trading_idx

    # attach selected metadata fields
    for col in ["file_name", "call_datetime_et", "after_market_close_et"]:
        if col in event_matches.columns:
            window_df[col] = event[col]

    event_windows.append(window_df)

event_windows = pd.concat(event_windows, ignore_index=True)

print("event_windows shape:", event_windows.shape)
display(event_windows.head(15))

event_windows shape: (24628, 13)


,Date,ticker,Adj Close,return,trading_idx,index_adj_close,market_return,event_id,event_trading_day_final,t,file_name,call_datetime_et,after_market_close_et
0,2015-08-05,AAPL,25.767361,0.006629,25,5139.939941,0.006736,1,2016-01-27,-120,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True
1,2015-08-06,AAPL,25.823441,0.002176,26,5056.439941,-0.016245,1,2016-01-27,-119,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True
2,2015-08-07,AAPL,25.910908,0.003387,27,5043.540039,-0.002551,1,2016-01-27,-118,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True
3,2015-08-10,AAPL,26.852962,0.036357,28,5101.799805,0.011551,1,2016-01-27,-117,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True
4,2015-08-11,AAPL,25.455584,-0.052038,29,5036.790039,-0.012743,1,2016-01-27,-116,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True
5,2015-08-12,AAPL,25.848112,0.015420,30,5044.390137,0.001509,1,2016-01-27,-115,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True
6,2015-08-13,AAPL,25.827919,-0.000781,31,5033.560059,-0.002147,1,2016-01-27,-114,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True
7,2015-08-14,AAPL,26.009600,0.007034,32,5048.240234,0.002916,1,2016-01-27,-113,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True
8,2015-08-17,AAPL,26.278763,0.010349,33,5091.700195,0.008609,1,2016-01-27,-112,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True
9,2015-08-18,AAPL,26.130724,-0.005633,34,5059.350098,-0.006353,1,2016-01-27,-111,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True


## 12. Check relative-time coverage

The theoretical full window length is:

`10 - (-120) + 1 = 131` trading days per event

Some events may have fewer rows if the ticker does not have enough pre-event history near the beginning of the sample.

In [13]:
window_counts = (
    event_windows.groupby("event_id")
    .size()
    .reset_index(name="n_rows")
)

window_counts["expected_full_window_rows"] = WINDOW_END - WINDOW_START + 1
window_counts["has_full_window"] = (
    window_counts["n_rows"] == window_counts["expected_full_window_rows"]
)

print(window_counts["has_full_window"].value_counts(dropna=False))
display(window_counts.head())

has_full_window
True    188
Name: count, dtype: int64


,event_id,n_rows,expected_full_window_rows,has_full_window
0,1,131,131,True
1,2,131,131,True
2,3,131,131,True
3,4,131,131,True
4,5,131,131,True


## 13. Inspect events with incomplete windows

This is important because market-model estimation requires enough pre-event data in the `[-120,-20]` window.

In [14]:
incomplete_windows = window_counts.loc[~window_counts["has_full_window"]].copy()

print("Events with incomplete full window:", len(incomplete_windows))
display(incomplete_windows.head(20))

Events with incomplete full window: 0


,event_id,n_rows,expected_full_window_rows,has_full_window


## 14. Validate event-time indexing

For each event:
- `t = 0` should exist exactly once
- `t` should increase in steps of 1 across trading days

In [15]:
t0_counts = (
    event_windows[event_windows["t"] == 0]
    .groupby("event_id")
    .size()
    .reset_index(name="t0_count")
)

print("Minimum t=0 count:", t0_counts["t0_count"].min())
print("Maximum t=0 count:", t0_counts["t0_count"].max())

assert (t0_counts["t0_count"] == 1).all(), "Each event should have exactly one t=0 row"

Minimum t=0 count: 1
Maximum t=0 count: 1


## 15. Check that all required analysis windows can be formed

The report requires the following windows:
- estimation: `[-120,-20]`
- CAR: `[0,1]` and `[0,3]`
- pre-volatility: `[-10,-1]`
- post-volatility: `[+1,+10]`

Here we check whether each event has complete coverage for those windows. :contentReference[oaicite:1]{index=1}

In [16]:
required_t_values = {
    "estimation_window": list(range(-120, -19)),   # -120 ... -20
    "car_01_window": list(range(0, 2)),            # 0, 1
    "car_03_window": list(range(0, 4)),            # 0, 1, 2, 3
    "pre_vol_window": list(range(-10, 0)),         # -10 ... -1
    "post_vol_window": list(range(1, 11)),         # 1 ... 10
}

event_t_sets = (
    event_windows.groupby("event_id")["t"]
    .apply(set)
    .to_dict()
)

coverage_rows = []

for event_id, t_set in event_t_sets.items():
    row = {"event_id": event_id}
    for window_name, required_ts in required_t_values.items():
        row[window_name] = set(required_ts).issubset(t_set)
    coverage_rows.append(row)

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df.head())

,event_id,estimation_window,car_01_window,car_03_window,pre_vol_window,post_vol_window
0,1,True,True,True,True,True
1,2,True,True,True,True,True
2,3,True,True,True,True,True
3,4,True,True,True,True,True
4,5,True,True,True,True,True


## 16. Summarize window coverage

This tells us how many events are eligible for each later analysis step.

In [17]:
coverage_summary = coverage_df.drop(columns=["event_id"]).sum().reset_index()
coverage_summary.columns = ["window", "n_events_with_full_coverage"]
coverage_summary["total_events"] = len(coverage_df)
coverage_summary["share"] = (
    coverage_summary["n_events_with_full_coverage"] / coverage_summary["total_events"]
)

display(coverage_summary)

,window,n_events_with_full_coverage,total_events,share
0,estimation_window,188,188,1.0
1,car_01_window,188,188,1.0
2,car_03_window,188,188,1.0
3,pre_vol_window,188,188,1.0
4,post_vol_window,188,188,1.0


## 17. Merge coverage flags back onto event metadata

This creates an event-level eligibility table for later notebooks.

In [18]:
event_eligibility = event_matches.merge(
    coverage_df,
    on="event_id",
    how="left",
    validate="one_to_one",
)

display(event_eligibility.head())

,ticker,file_name,event_trading_day_final,call_datetime_et,after_market_close_et,event_id,Date,trading_idx,estimation_window,car_01_window,car_03_window,pre_vol_window,post_vol_window
0,AAPL,2016-Jan-26-AAPL.txt,2016-01-27,2016-01-26T17:00:00-05:00,True,1,2016-01-27,145,True,True,True,True,True
1,AAPL,2016-Apr-26-AAPL.txt,2016-04-27,2016-04-26T17:00:00-04:00,True,2,2016-04-27,208,True,True,True,True,True
2,AAPL,2016-Jul-26-AAPL.txt,2016-07-27,2016-07-26T17:00:00-04:00,True,3,2016-07-27,271,True,True,True,True,True
3,AAPL,2016-Oct-25-AAPL.txt,2016-10-26,2016-10-25T17:00:00-04:00,True,4,2016-10-26,335,True,True,True,True,True
4,AAPL,2017-Jan-31-AAPL.txt,2017-02-01,2017-01-31T17:00:00-05:00,True,5,2017-02-01,401,True,True,True,True,True


## 18. Finalize event-window dataset

We keep only the columns needed for downstream analysis.

In [19]:
final_cols = [
    "event_id",
    "ticker",
    "file_name",
    "call_datetime_et",
    "after_market_close_et",
    "event_trading_day_final",
    "Date",
    "t",
    "Adj Close",
    "return",
    "index_adj_close",
    "market_return",
]

final_cols = [c for c in final_cols if c in event_windows.columns]
event_windows = event_windows[final_cols].copy()

event_windows = event_windows.sort_values(["event_id", "t"]).reset_index(drop=True)

display(event_windows.head(20))

,event_id,ticker,file_name,call_datetime_et,after_market_close_et,event_trading_day_final,Date,t,Adj Close,return,index_adj_close,market_return
0,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-05,-120,25.767361,0.006629,5139.939941,0.006736
1,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-06,-119,25.823441,0.002176,5056.439941,-0.016245
2,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-07,-118,25.910908,0.003387,5043.540039,-0.002551
3,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-10,-117,26.852962,0.036357,5101.799805,0.011551
4,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-11,-116,25.455584,-0.052038,5036.790039,-0.012743
5,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-12,-115,25.848112,0.015420,5044.390137,0.001509
6,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-13,-114,25.827919,-0.000781,5033.560059,-0.002147
7,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-14,-113,26.009600,0.007034,5048.240234,0.002916
8,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-17,-112,26.278763,0.010349,5091.700195,0.008609
9,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-18,-111,26.130724,-0.005633,5059.350098,-0.006353


## 19. Save outputs

We save:
- the full event-time panel
- the event-level window coverage table

In [20]:
event_windows.to_csv("../data/processed/event_windows.csv", index=False)
event_eligibility.to_csv("../data/processed/event_window_eligibility.csv", index=False)

print("Saved:")
print("../data/processed/event_windows.csv")
print("../data/processed/event_window_eligibility.csv")

Saved:
../data/processed/event_windows.csv
../data/processed/event_window_eligibility.csv


## 20. Validation summary

This summary reports:
- number of events
- number of event-time rows
- number of unmatched events
- number of events with complete estimation and event windows

In [21]:
summary = {
    "n_events": len(event_matches),
    "n_event_time_rows": len(event_windows),
    "n_unmatched_events": len(missing_event_matches),
    "n_events_full_estimation_window": int(coverage_df["estimation_window"].sum()),
    "n_events_full_car_01_window": int(coverage_df["car_01_window"].sum()),
    "n_events_full_car_03_window": int(coverage_df["car_03_window"].sum()),
    "n_events_full_pre_vol_window": int(coverage_df["pre_vol_window"].sum()),
    "n_events_full_post_vol_window": int(coverage_df["post_vol_window"].sum()),
}

summary_df = pd.DataFrame(summary.items(), columns=["check", "value"])
display(summary_df)

,check,value
0,n_events,188
1,n_event_time_rows,24628
2,n_unmatched_events,0
3,n_events_full_estimation_window,188
4,n_events_full_car_01_window,188
5,n_events_full_car_03_window,188
6,n_events_full_pre_vol_window,188
7,n_events_full_post_vol_window,188


all 188 events matched successfully to stock return data

the event-time panel contains 24,628 rows, which is exactly 188 × 131

there were 0 unmatched events

all 188 events have complete coverage for:

estimation window [-120,-20]

CAR window [0,1]

CAR window [0,3]

pre-volatility window [-10,-1]

post-volatility window [+1,+10]